# 01 — Data Cleaning
Loads raw CSV, cleans it, and saves `cases_clean.csv`.

In [ ]:
# Install dependencies (run once)
!pip install pandas -q

In [ ]:
import pandas as pd
import re
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
RAW_PATH     = Path('data/raw/cases_raw.csv')
CLEAN_PATH   = Path('data/processed/cases_clean.csv')
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Load ───────────────────────────────────────────────────────────────────
df = pd.read_csv(RAW_PATH, low_memory=False)
print(f'Loaded {len(df):,} rows | columns: {list(df.columns)}')

In [ ]:
# ── Column mapping ─────────────────────────────────────────────────────────
# Actual columns: diary_no, Judgement_type, case_no, pet, res,
#                 pet_adv, res_adv, bench, judgement_by,
#                 judgment_dates, temp_link, language

KEEP = {
    'diary_no'       : 'case_id',
    'case_no'        : 'case_no',
    'pet'            : 'petitioner',
    'res'            : 'respondent',
    'bench'          : 'bench',
    'judgement_by'   : 'judge',
    'judgment_dates' : 'decision_date',
    'temp_link'      : 'pdf_path',
    'Judgement_type' : 'judgement_type',
}

df = df.rename(columns=KEEP)[list(KEEP.values())]
print('Kept columns:', list(df.columns))

In [ ]:
# ── Clean text fields ──────────────────────────────────────────────────────
def clean_text(s):
    if not isinstance(s, str):
        return ''
    s = re.sub(r'\s+', ' ', s)   # collapse whitespace
    return s.strip()

text_cols = ['petitioner', 'respondent', 'bench', 'judge', 'case_no']
for col in text_cols:
    df[col] = df[col].apply(clean_text)

# ── Build case_text ────────────────────────────────────────────────────────
df['case_text'] = (
    'Case: '       + df['case_no']    + ' | ' +
    'Petitioner: ' + df['petitioner'] + ' | ' +
    'Respondent: ' + df['respondent'] + ' | ' +
    'Judge: '      + df['judge']      + ' | ' +
    'Bench: '      + df['bench']
)
print('case_text sample:')
print(df['case_text'].iloc[0])

In [ ]:
# ── Parse decision year ────────────────────────────────────────────────────
def extract_year(s):
    if not isinstance(s, str):
        return None
    m = re.search(r'(\d{4})', s)
    return int(m.group(1)) if m else None

df['year'] = df['decision_date'].apply(extract_year)
print(df['year'].value_counts().sort_index())

In [ ]:
# ── Validate PDF links (keep only rows with a non-empty path) ───────────────
df['pdf_path'] = df['pdf_path'].apply(lambda x: x.strip() if isinstance(x, str) else '')

# Drop rows with no PDF link
before = len(df)
df = df[df['pdf_path'] != ''].reset_index(drop=True)
print(f'Dropped {before - len(df):,} rows with missing PDF → {len(df):,} remaining')

In [ ]:
# ── Save ───────────────────────────────────────────────────────────────────
df.to_csv(CLEAN_PATH, index=False)
print(f'Saved → {CLEAN_PATH}  ({len(df):,} rows)')
df.head(3)